# 第六课:Self-Attention 手撕

讲义:[06-self-attention.md](06-self-attention.md)。"它"指谁 → 点积打分 → softmax 加权 → 搬运信息。

## 1. 纯 Python:一个词一个词算(讲义 §6)

玩具向量(手工设定):「它」被设计成和「小猫」很像。

In [ ]:
import math

def softmax(x):
    max_x = max(x)
    exp_x = [math.exp(v - max_x) for v in x]
    total = sum(exp_x)
    return [v / total for v in exp_x]

def dot(a, b):
    return sum(ai * bi for ai, bi in zip(a, b))

def self_attention(words, X):
    d = len(X[0])
    outs = []
    for i, w in enumerate(words):
        q = X[i]                                          # ① q 和每个 k 点积
        scores = [dot(q, k) / math.sqrt(d) for k in X]    # ② 除以 √d_k
        weights = softmax(scores)                         # ③ 变成概率
        out = [0.0] * d                                   # ④ 加权搬运 v
        for wgt, v in zip(weights, X):
            out = [o + wgt * vi for o, vi in zip(out, v)]
        outs.append(out)
        print(f'{w}: weights = {[round(x, 3) for x in weights]}')
    return outs

words = ['小猫', '它', '可爱']
X = [[1.0, 1.0, 0.0, 0.0],
     [0.9, 1.0, 0.0, 0.0],
     [0.0, 0.0, 1.0, 1.0]]

outs = self_attention(words, X)
print('「它」的新向量 =', [round(v, 3) for v in outs[1]])

## 2. 矩阵版:一次算完所有词(讲义 §8)

W 矩阵 = attention 热力图,每行和 = 1。

In [ ]:
import numpy as np

def softmax_rows(M):
    m = M.max(axis=1, keepdims=True)
    e = np.exp(M - m)
    return e / e.sum(axis=1, keepdims=True)

def self_attention_matrix(X):
    n, d = len(X), len(X[0])
    scores = X @ X.T / math.sqrt(d)      # ①② 所有配对打分
    W = softmax_rows(scores)             # ③ 每行 softmax(dim=-1!)
    out = W @ X                          # ④ 加权搬运
    return W, out

Xnp = np.array(X)
W, out = self_attention_matrix(Xnp)
print('W =\n', np.round(W, 3))
print('每行和 =', np.round(W.sum(axis=1), 6))
print('out =\n', np.round(out, 3))

## 3. √d_k 实验:不缩放会怎样(讲义 §9)

d=64 随机向量:不除,分数天然 ±8 量级,softmax 一家独大;除完,恢复平滑。

In [ ]:
rng = np.random.default_rng(42)
d = 64
q  = rng.normal(size=d)
ks = rng.normal(size=(5, d))

raw    = ks @ q
scaled = raw / math.sqrt(d)

print('raw    ', np.round(raw, 1),    '->', np.round(softmax_rows(raw.reshape(1, -1))[0], 3))
print('scaled ', np.round(scaled, 2), '->', np.round(softmax_rows(scaled.reshape(1, -1))[0], 3))

## 作业现场

- C2:给 `self_attention_matrix` 加 `causal=True`:上三角填 `-np.inf`。
- C3 验证:掩码后 W 的上三角全 0,每行和仍为 1。
- E1:5 词 6 维,自设语义向量,打印 W 当热力图。

In [ ]:
# TODO(C2): 因果掩码版
def self_attention_causal(X):
    # 你的代码(提示:np.triu(ones, k=1) 生成上三角掩码;scores[掩码] = -np.inf)
    ...

# TODO(C3): 验证
# W = self_attention_causal(Xnp)[0]
# print('上三角和 =', np.triu(W, k=1).sum())      # 期望 0.0
# print('每行和   =', np.round(W.sum(axis=1), 6))  # 期望全 1

# TODO(E1): 5 词自设向量热力图
# sentences = ['银行', '的', '门口', '很', '安静']  ← 自己设计向量
